In [ ]:
import os
import sys
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# --- 1. SYSTEM SETUP ---
root_dir = os.path.abspath("..")
paths_to_add = [
    root_dir,
    os.path.abspath("../current_setpoints"),
    os.path.abspath("../current_setpoints/optimization"),
    os.path.abspath("../current_setpoints/model")
]
for path in paths_to_add:
    if path not in sys.path:
        sys.path.append(path)

from current_setpoints.data import FluxValues, IEEEMachine2
from current_setpoints.utils import NeuralTorquePredictor, train_model, load_aggregated_csv_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {DEVICE}")

# --- 2. PREPARE TRAINING DATA & FIT SCALER ---
data = load_aggregated_csv_data('../data/aggregated_file_means.csv', 
                                {'omega': 'omega', 'id1': 'id1', 'iq1': 'iq1', 'id3': 'id3', 'iq3': 'iq3', 'torq': 'torq'})

X = data[['omega', 'id1', 'iq1', 'id3', 'iq3']].values.astype(np.float32)
y = data[['torq']].values.astype(np.float32)

flux_values = FluxValues()
machine = IEEEMachine2(flux_values) # IMPORTANT: Ensure the 2* error is FIXED in this class!

B_list = []
for row in X:
    machine.update_state(omega=row[0], vec_curr_dq=row[1:])
    B_list.append(machine.vec_b.copy())
B_array = np.array(B_list).astype(np.float32)

X_train_val, X_test, y_train_val, y_test, B_train_val, B_test = train_test_split(
    X, y, B_array, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val, B_train, B_val = train_test_split(
    X_train_val, y_train_val, B_train_val, test_size=0.20, random_state=42
)

scaler_X = StandardScaler()
X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm = scaler_X.transform(X_val)

# --- 3. EXTRACT THE "SIMILARITY DATASET" FROM PICKLE ---
print("Extracting full grid from pickle file for similarity comparison...")
PICKLE_FILE_PATH = "../data/IEEETIE_machine2_correction.pkl" # REPLACE WITH YOUR FILENAME

with open(PICKLE_FILE_PATH, 'rb') as f:
    pkl_data = pickle.load(f)

sim_inputs_raw = []
sim_targets = []

# Flatten the 201x201 grid and discard NaNs
for i in range(201):
    for j in range(201):
        t_val = pkl_data['T_pirn_matrix'][i, j]
        if not np.isnan(t_val):
            om = pkl_data['om_vec'][j]
            id1 = pkl_data['isd1'][i, j]
            iq1 = pkl_data['isq1'][i, j]
            id3 = pkl_data['isd3'][i, j]
            iq3 = pkl_data['isq3'][i, j]
            
            sim_inputs_raw.append([om, id1, iq1, id3, iq3])
            sim_targets.append(t_val)

sim_inputs_raw = np.array(sim_inputs_raw, dtype=np.float32)
sim_targets_tensor = torch.tensor(sim_targets, dtype=torch.float32).to(DEVICE)

print(f"Extracted {len(sim_inputs_raw)} valid grid points for comparison.")

# Pre-calculate B vectors for the similarity dataset so we don't do it every loop
print("Pre-calculating B-vectors for the similarity dataset (this takes a moment)...")
sim_B_list = []
for row in sim_inputs_raw:
    machine.update_state(omega=row[0], vec_curr_dq=row[1:])
    sim_B_list.append(machine.vec_b.copy())

sim_inputs_norm = scaler_X.transform(sim_inputs_raw)
sim_inputs_tensor = torch.tensor(sim_inputs_norm, dtype=torch.float32).to(DEVICE)
sim_B_tensor = torch.tensor(np.array(sim_B_list), dtype=torch.float32).to(DEVICE)

# --- 4. THE SIMILARITY SWEEP ---
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# We don't need 10,000. 100-200 seeds is usually plenty to find a very close match.
seeds_to_try = [42, 0, 1, 123, 1234, 12345, 999] + list(range(10000, 50000)) #0.0206
best_overall_error = float('inf')
best_seed = None

print("\nStarting the Similarity Sweep...")
print("Goal: Find the model that visually matches the old plots the best.")

for seed in seeds_to_try:
    seed_everything(seed)
    
    g = torch.Generator()
    g.manual_seed(seed)
    
    train_dataset = TensorDataset(torch.from_numpy(X_train_norm).float(), torch.from_numpy(y_train).float(), torch.from_numpy(B_train).float())
    val_dataset = TensorDataset(torch.from_numpy(X_val_norm).float(), torch.from_numpy(y_val).float(), torch.from_numpy(B_val).float())
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    model = NeuralTorquePredictor(
        input_size=5, hidden_size=12, scaler_X=scaler_X, machine=machine, device=DEVICE
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-02, weight_decay=1.0e-04)
    criterion = nn.MSELoss()
    
    # Train the model
    train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        criterion=criterion, optimizer=optimizer,
        num_epochs=800, patience=15, device=DEVICE, min_delta=1e-5, verbose=False
    )
    
    # Evaluate Similarity against the old grid
    model.eval()
    with torch.no_grad():
        try:
            preds = model(sim_inputs_tensor, sim_B_tensor)
        except TypeError:
            preds = model(sim_inputs_tensor) 
            
        # Calculate Mean Absolute Error (MAE) across the entire grid
        current_error = torch.mean(torch.abs(preds.squeeze() - sim_targets_tensor)).item()
        
    if current_error < best_overall_error:
        best_overall_error = current_error
        best_seed = seed
        print(f"\n🌟 NEW BEST MATCH! Seed: {seed} | Average Torque Diff: {current_error:.4f} Nm")
        torch.save(model.state_dict(), "similarity_best_model.pth")
        
    if seed % 5 == 0:
        print(f"Sweeping... Currently on seed {seed}. (Best so far: Seed {best_seed} with {best_overall_error:.4f} error)", end="\r")

print(f"\n\n✅ Sweep Complete! Best visual match is Seed {best_seed} with an average grid difference of {best_overall_error:.4f} Nm.")
print("The weights for this seed have been saved as 'similarity_best_model.pth'.")

In [3]:
import os
import sys
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# --- 1. SYSTEM SETUP ---
root_dir = os.path.abspath("..")
paths_to_add = [
    root_dir,
    os.path.abspath("../current_setpoints"),
    os.path.abspath("../current_setpoints/optimization"),
    os.path.abspath("../current_setpoints/model")
]
for path in paths_to_add:
    if path not in sys.path:
        sys.path.append(path)

from current_setpoints.data import FluxValues, IEEEMachine2
from current_setpoints.utils import NeuralTorquePredictor, train_model, load_aggregated_csv_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {DEVICE}")

# --- 2. PREPARE TRAINING DATA & FIT SCALER ---
data = load_aggregated_csv_data('../data/aggregated_file_means.csv', 
                                {'omega': 'omega', 'id1': 'id1', 'iq1': 'iq1', 'id3': 'id3', 'iq3': 'iq3', 'torq': 'torq'})

X = data[['omega', 'id1', 'iq1', 'id3', 'iq3']].values.astype(np.float32)
y = data[['torq']].values.astype(np.float32)

flux_values = FluxValues()
machine = IEEEMachine2(flux_values) # IMPORTANT: Ensure the 2* error is FIXED in this class!

B_list = []
for row in X:
    machine.update_state(omega=row[0], vec_curr_dq=row[1:])
    B_list.append(machine.vec_b.copy())
B_array = np.array(B_list).astype(np.float32)

X_train_val, X_test, y_train_val, y_test, B_train_val, B_test = train_test_split(
    X, y, B_array, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val, B_train, B_val = train_test_split(
    X_train_val, y_train_val, B_train_val, test_size=0.20, random_state=42
)

scaler_X = StandardScaler()
X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm = scaler_X.transform(X_val)

# --- 3. EXTRACT THE "SIMILARITY DATASET" FROM PICKLE ---
print("Extracting full grid from pickle file for similarity comparison...")
PICKLE_FILE_PATH = "../data/IEEETIE_machine2_correction.pkl" # REPLACE WITH YOUR FILENAME

with open(PICKLE_FILE_PATH, 'rb') as f:
    pkl_data = pickle.load(f)

sim_inputs_raw = []
sim_targets = []

# Flatten the 201x201 grid and discard NaNs
for i in range(201):
    for j in range(201):
        t_val = pkl_data['T_pirn_matrix'][i, j]
        if not np.isnan(t_val):
            om = pkl_data['om_vec'][j]
            id1 = pkl_data['isd1'][i, j]
            iq1 = pkl_data['isq1'][i, j]
            id3 = pkl_data['isd3'][i, j]
            iq3 = pkl_data['isq3'][i, j]
            
            sim_inputs_raw.append([om, id1, iq1, id3, iq3])
            sim_targets.append(t_val)

sim_inputs_raw = np.array(sim_inputs_raw, dtype=np.float32)
sim_targets_tensor = torch.tensor(sim_targets, dtype=torch.float32).to(DEVICE)

print(f"Extracted {len(sim_inputs_raw)} valid grid points for comparison.")

# Pre-calculate B vectors for the similarity dataset so we don't do it every loop
print("Pre-calculating B-vectors for the similarity dataset (this takes a moment)...")
sim_B_list = []
for row in sim_inputs_raw:
    machine.update_state(omega=row[0], vec_curr_dq=row[1:])
    sim_B_list.append(machine.vec_b.copy())

sim_inputs_norm = scaler_X.transform(sim_inputs_raw)
sim_inputs_tensor = torch.tensor(sim_inputs_norm, dtype=torch.float32).to(DEVICE)
sim_B_tensor = torch.tensor(np.array(sim_B_list), dtype=torch.float32).to(DEVICE)

# Extract the true maximum torque from the target dataset
TARGET_PEAK_TORQUE = sim_targets_tensor.max().item()
print(f"Target Peak Torque to match: {TARGET_PEAK_TORQUE:.4f} Nm")

# --- 4. THE SIMILARITY SWEEP ---
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seeds_to_try = [42, 0, 1, 123, 1234, 12345, 999] + list(range(30183, 30185))
best_overall_score = float('inf')
best_seed = None
best_peak_diff = float('inf')
best_mean_diff = float('inf')

print("\nStarting the Similarity Sweep...")
print("Goal: Find the model that matches the PEAK torque first, and average shape second.")

for seed in seeds_to_try:
    seed_everything(seed)
    
    g = torch.Generator()
    g.manual_seed(seed)
    
    train_dataset = TensorDataset(torch.from_numpy(X_train_norm).float(), torch.from_numpy(y_train).float(), torch.from_numpy(B_train).float())
    val_dataset = TensorDataset(torch.from_numpy(X_val_norm).float(), torch.from_numpy(y_val).float(), torch.from_numpy(B_val).float())
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    model = NeuralTorquePredictor(
        input_size=5, hidden_size=12, scaler_X=scaler_X, machine=machine, device=DEVICE
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-02, weight_decay=1.0e-04)
    criterion = nn.MSELoss()
    
    # Train the model
    train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        criterion=criterion, optimizer=optimizer,
        num_epochs=800, patience=15, device=DEVICE, min_delta=1e-5, verbose=False
    )
    
    # Evaluate Similarity against the old grid
    model.eval()
    with torch.no_grad():
        try:
            preds = model(sim_inputs_tensor, sim_B_tensor).squeeze()
        except TypeError:
            preds = model(sim_inputs_tensor).squeeze()
            
        # 1. Calculate Average Error
        mean_error = torch.mean(torch.abs(preds - sim_targets_tensor)).item()
        
        # 2. Calculate Peak Torque Error
        pred_peak = preds.max().item()
        peak_error = abs(TARGET_PEAK_TORQUE - pred_peak)
        
        # 3. Create a weighted score (1 Nm of peak error is penalized 100x more than 1 Nm of average error)
        combined_score = (peak_error * 100) + mean_error
        
    if combined_score < best_overall_score:
        best_overall_score = combined_score
        best_seed = seed
        best_peak_diff = peak_error
        best_mean_diff = mean_error
        print(f"\n🌟 NEW BEST MATCH! Seed: {seed}")
        print(f"   -> Peak Torque Diff:  {peak_error:.4f} Nm (Target: {TARGET_PEAK_TORQUE:.2f}, Pred: {pred_peak:.2f})")
        print(f"   -> Average Grid Diff: {mean_error:.4f} Nm")
        torch.save(model.state_dict(), "similarity_best_model.pth")
        
    if seed % 5 == 0:
        print(f"Sweeping... Currently on seed {seed}. (Best Seed: {best_seed} | Peak Diff: {best_peak_diff:.4f})", end="\r")

print(f"\n\n✅ Sweep Complete! Best visual match is Seed {best_seed}.")
print(f"Final Stats - Peak Torque Difference: {best_peak_diff:.4f} Nm | Average Grid Difference: {best_mean_diff:.4f} Nm")
print("The weights for this seed have been saved as 'similarity_best_model.pth'.")

Using compute device: cuda
Loaded 174 valid data points from CSV.
Extracting full grid from pickle file for similarity comparison...
Extracted 35798 valid grid points for comparison.
Pre-calculating B-vectors for the similarity dataset (this takes a moment)...
Target Peak Torque to match: 7.8268 Nm

Starting the Similarity Sweep...
Goal: Find the model that matches the PEAK torque first, and average shape second.

Starting Training with Early Stopping (Patience=15)...

Early stopping triggered at epoch 120!

🌟 NEW BEST MATCH! Seed: 42
   -> Peak Torque Diff:  0.1610 Nm (Target: 7.83, Pred: 7.99)
   -> Average Grid Diff: 0.0564 Nm

Starting Training with Early Stopping (Patience=15)...

Early stopping triggered at epoch 131!

🌟 NEW BEST MATCH! Seed: 0
   -> Peak Torque Diff:  0.1341 Nm (Target: 7.83, Pred: 7.96)
   -> Average Grid Diff: 0.0393 Nm
Sweeping... Currently on seed 0. (Best Seed: 0 | Peak Diff: 0.1341)
Starting Training with Early Stopping (Patience=15)...

Early stopping tri

In [6]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- 1. PATH SETUP ---
root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.data import FluxValues, IEEEMachine2
from current_setpoints.utils import NeuralTorquePredictor, load_aggregated_csv_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "../weights/NTM_Best_Model.pth"
DATA_PATH = '../data/aggregated_file_means.csv'
SCALER_PATH = '../weights/NTM_Best_Scaler.npy' # Ensure this points to your original scaler file

# --- 2. LOAD EXPERIMENTAL DATA ---
print(f"Loading experimental data from {DATA_PATH}...")
data = load_aggregated_csv_data(DATA_PATH, 
                                {'omega': 'omega', 'id1': 'id1', 'iq1': 'iq1', 'id3': 'id3', 'iq3': 'iq3', 'torq': 'torq'})

X = data[['omega', 'id1', 'iq1', 'id3', 'iq3']].values.astype(np.float32)
y = data[['torq']].values.astype(np.float32)

# --- 3. RECREATE SPLIT & LOAD ORIGINAL SCALER ---
# Use random_state=42 to match your training script's data separation
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

print(f"Loading original scaler from {SCALER_PATH}...")
# Assuming the .npy was saved using np.save() on a dictionary or the object itself
scaler_data = np.load(SCALER_PATH, allow_pickle=True).item()

scaler_X = StandardScaler()
# Manually inject the original mean and scale
scaler_X.mean_ = scaler_data['mean']
scaler_X.scale_ = scaler_data['scale']
# Standardscaler also expects 'var_' and 'n_samples_seen_' usually, 
# but transform() only strictly requires mean_ and scale_.

X_test_norm = scaler_X.transform(X_test)

# --- 4. PHYSICS VECTOR CALCULATION ---
flux_values = FluxValues()
machine = IEEEMachine2(flux_values) # Your fixed physics class

B_test_list = []
for row in X_test:
    machine.update_state(omega=row[0], vec_curr_dq=row[1:])
    B_test_list.append(machine.vec_b.copy())
B_test_tensor = torch.from_numpy(np.array(B_test_list).astype(np.float32)).to(DEVICE)
X_test_tensor = torch.from_numpy(X_test_norm).to(DEVICE)

# --- 5. MODEL INFERENCE ---
model = NeuralTorquePredictor(
    input_size=5, hidden_size=12, scaler_X=scaler_X, machine=machine, device=DEVICE
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

with torch.no_grad():
    predictions = model(X_test_tensor, B_test_tensor).cpu().numpy().flatten()

# --- 6. METRICS & DIAGNOSTICS ---
targets = y_test.flatten()
errors = predictions - targets
rmse = np.sqrt(np.mean(errors**2))
mae = np.mean(np.abs(errors))
bias = np.mean(errors) # Tells us if the model is systematically high or low

print("\n" + "="*40)
print("       REFECTORED MODEL EVALUATION")
print("="*40)
print(f"RMSE:               {rmse:.5f} Nm")
print(f"MAE:                {mae:.5f} Nm")
print(f"Systematic Bias:    {bias:.5f} Nm")
print("-" * 40)
print("Sample Comparison (Target vs Prediction):")
for i in range(min(5, len(targets))):
    print(f"  Target: {targets[i]:.4f} | Pred: {predictions[i]:.4f} | Diff: {errors[i]:.4f}")
print("="*40)

Loading experimental data from ../data/aggregated_file_means.csv...
Loaded 174 valid data points from CSV.
Loading original scaler from ../weights/NTM_Best_Scaler.npy...

       REFECTORED MODEL EVALUATION
RMSE:               0.04609 Nm
MAE:                0.03309 Nm
Systematic Bias:    -0.00644 Nm
----------------------------------------
Sample Comparison (Target vs Prediction):
  Target: 3.8240 | Pred: 3.8264 | Diff: 0.0024
  Target: 3.0660 | Pred: 3.0564 | Diff: -0.0096
  Target: 4.6541 | Pred: 4.7095 | Diff: 0.0554
  Target: 0.8675 | Pred: 0.8624 | Diff: -0.0050
  Target: 4.0588 | Pred: 4.0438 | Diff: -0.0150
